# Impulse — A2D2 Bus-Signal & Camera Ingestion

Ingest one recording (`20190401_121727`) of the **A2D2 (Audi Autonomous Driving Dataset)**
into the Impulse silver-layer model, and **untar the full front-center camera archive in
parallel** (PNG + `cam_tstamp`). All frames are written to the volume; frames inside the bus
time window are registered in `*_camera_frames` so they can be matched to the bus signals.

## ⚠️ Dataset license — read before running
A2D2 is published by **Audi AG** under **CC BY-ND 4.0** (NoDerivatives):
<https://creativecommons.org/licenses/by-nd/4.0/> · source <https://www.a2d2.audi/> ·
paper arXiv:2004.06320. Because of the **NoDerivatives** clause this repository contains
**no A2D2 data**: this notebook downloads it **at runtime, from the official source, into
your own workspace**, and writes all derived tables/images into **your own** Unity Catalog.
**Do not commit any downloaded or derived data**, and commit this notebook with outputs
cleared.

**Requirements:** Unity Catalog; serverless or DBR 14+ (Spark executors write PNGs to the
`/Volumes` FUSE mount); privilege to create a schema and a volume; internet egress to AWS S3
(`eu-central-1`, Range requests supported); **~98 GB of volume free space** for the full
extracted front-center archive.

In [ ]:
%pip install pydantic>=2.0 scipy -q
dbutils.library.restartPython()

In [ ]:
BASE = "https://aev-autonomous-driving-dataset.s3.eu-central-1.amazonaws.com"
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("schema", "", "Schema")
dbutils.widgets.text("table_prefix", "a2d2", "Table Prefix")
dbutils.widgets.text("volume", "a2d2_raw", "UC Volume (runtime download)")
# One drive per run -> one container. Defaults describe the Munich drive (container 1).
dbutils.widgets.text("container_id", "1", "Container id for this drive")
dbutils.widgets.text("city", "Munich", "City of this test drive")
dbutils.widgets.text("recording", "20190401_121727", "Recording id (date_time)")
dbutils.widgets.text("bus_url", f"{BASE}/camera_lidar-20190401121727_bus_signals.tar", "Bus-signals tar URL")
dbutils.widgets.text("cam_url", f"{BASE}/camera_lidar-20190401121727_camera_frontcenter.tar", "Front-camera tar URL")
dbutils.widgets.dropdown("download_images", "true", ["true", "false"], "Download camera images")
dbutils.widgets.text("extract_partitions", "64", "Parallel Spark tasks for the camera-tar untar")
dbutils.widgets.dropdown("skip_download_if_present", "true", ["true", "false"], "Skip download if present")
dbutils.widgets.dropdown("drop_created_tables", "false", ["true", "false"], "Drop created tables")

In [ ]:
import sys, os

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
TABLE_PREFIX = dbutils.widgets.get("table_prefix")
VOLUME = dbutils.widgets.get("volume")
DOWNLOAD_IMAGES = dbutils.widgets.get("download_images") == "true"
EXTRACT_PARTITIONS = max(1, int(dbutils.widgets.get("extract_partitions") or "64"))
SKIP_IF_PRESENT = dbutils.widgets.get("skip_download_if_present") == "true"
CONTAINER_ID = int(dbutils.widgets.get("container_id") or "1")
CITY = dbutils.widgets.get("city") or ""
RECORDING = dbutils.widgets.get("recording") or "20190401_121727"
BUS_URL = dbutils.widgets.get("bus_url")
CAM_URL = dbutils.widgets.get("cam_url")

if not (CATALOG and SCHEMA and TABLE_PREFIX and VOLUME):
    raise ValueError("Please set catalog, schema, table_prefix and volume widgets above.")

# When run from a cloned repo / Git folder, the notebook lives at demos/a2d2/<nb> so the
# repo's src/ is 3 path-segments up; add it to sys.path so the impulse packages import.
# When deployed as a job (e.g. via the DABs bundle in this folder) the impulse wheel is
# installed instead, so we only insert src/ when it actually exists.
nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
REPO_ROOT = "/Workspace" + "/".join(nb_path.split("/")[:-3])
src_dir = os.path.join(REPO_ROOT, "src")
if os.path.isdir(src_dir):
    sys.path.insert(0, src_dir)

pfx = f"{CATALOG}.{SCHEMA}.{TABLE_PREFIX}"
LICENSE_URL = "https://creativecommons.org/licenses/by-nd/4.0/"
print(f"Target tables: {pfx}_*   (container_id={CONTAINER_ID}, city={CITY}, recording={RECORDING}, download_images={DOWNLOAD_IMAGES})")

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

# Per-drive subfolder so the three drives' raw data + PNGs never collide.
vol_root = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
vol_root_c = f"{vol_root}/container_{CONTAINER_ID}"
bus_tar_path = f"{vol_root_c}/bus_signals.tar"
cam_dir = f"{vol_root_c}/camera_lidar/{RECORDING}/camera/cam_front_center"
os.makedirs(cam_dir, exist_ok=True)
print("Container volume root:", vol_root_c)

## Bus signals → silver-layer tables

Download the bus-signals tar (~185 MB), parse the 22 channels, and write the five
silver-layer tables. Channels are stored **RAW** (`container_id, channel_id, timestamp,
value`); the query engine run-length-encodes them on the fly (`data_type="RAW"`).

In [ ]:
import glob, requests

def stream_download(url, dest, chunk=8 * 1024 * 1024):
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    with requests.get(url, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(dest, "wb") as f:
            for c in r.iter_content(chunk_size=chunk):
                f.write(c)

already = SKIP_IF_PRESENT and (
    glob.glob(f"{vol_root_c}/**/*_bus_signals.json", recursive=True) or os.path.exists(bus_tar_path)
)
if already:
    print("Bus tar/JSON already present; skipping download.")
else:
    try:
        print("Downloading bus-signals tar (~185 MB) ...")
        stream_download(BUS_URL, bus_tar_path)
        print("Downloaded:", bus_tar_path)
    except Exception as e:
        raise RuntimeError(
            f"Could not download {BUS_URL} ({e}). If egress is blocked, download the tar "
            f"manually from https://www.a2d2.audi/ and place it at {bus_tar_path}."
        )

In [ ]:
import tarfile, glob

found = glob.glob(f"{vol_root_c}/**/*_bus_signals.json", recursive=True)
if SKIP_IF_PRESENT and found:
    bus_json_path = found[0]
    print("Using already-extracted bus JSON:", bus_json_path)
else:
    with tarfile.open(bus_tar_path) as t:
        member = next(m for m in t.getmembers() if m.name.endswith("_bus_signals.json"))
        t.extract(member, path=vol_root_c)
        bus_json_path = os.path.join(vol_root_c, member.name)
    print("Extracted bus JSON:", bus_json_path)

### Parse and build the silver-layer DataFrames
One **container** = this recording. Each of the 22 bus channels becomes a **channel** with
a deterministic `channel_id` (1..22, by sorted name). Per-channel statistics are computed
with the engine's own `SampleSeries` semantics for parity with query-time results.

In [ ]:
import json
import numpy as np

with open(bus_json_path) as f:
    raw = json.load(f)

channel_names = sorted(raw.keys())
assert len(channel_names) == 22, f"expected 22 channels, got {len(channel_names)}"
channel_meta = {name: {"channel_id": i + 1, "unit": raw[name].get("unit")}
                for i, name in enumerate(channel_names)}

print(f"Parsed {len(channel_names)} channels:")
for name in channel_names:
    cm = channel_meta[name]
    print(f"  [{cm['channel_id']:2d}] {name:38s} unit={str(cm['unit']):28s} n={len(raw[name]['values']):>7,}")

In [ ]:
import pandas as pd
from impulse_query_engine import schema as S

parts = []
for name in channel_names:
    arr = np.asarray(raw[name]["values"], dtype=np.float64)  # (n, 2): [ts_us, value]
    parts.append(pd.DataFrame({
        "container_id": np.int64(CONTAINER_ID),
        "channel_id": np.int32(channel_meta[name]["channel_id"]),
        "timestamp": arr[:, 0].astype(np.int64),
        "value": arr[:, 1].astype(np.float64),
    }))
channels_pdf = pd.concat(parts, ignore_index=True)
# RAW point storage; the DeltaSolver converts to RLE at query time. Do NOT dedupe here.
channels_df = spark.createDataFrame(channels_pdf, schema=S.CHANNELS_SCHEMA_WITHOUT_RLE)
print(f"channels rows: {len(channels_pdf):,}")

In [ ]:
# CHANNEL_TAGS (EAV). "channel_name" is required so query.channel(channel_name=...) resolves.
tag_rows = []
for name in channel_names:
    cid = channel_meta[name]["channel_id"]
    unit = channel_meta[name]["unit"]
    tag_rows.append((CONTAINER_ID, cid, "channel_name", name))
    tag_rows.append((CONTAINER_ID, cid, "unit", "" if unit is None else str(unit)))
channel_tags_df = spark.createDataFrame(tag_rows, schema=S.CHANNEL_TAGS)
print(f"channel_tags rows: {len(tag_rows)}")

In [ ]:
from impulse_query_engine.model.series.sample_series import SampleSeries

def weighted_std(values, w):
    wsum = w.sum()
    if wsum <= 0:
        return float("nan")
    m = np.nansum(values * w) / wsum
    return float(np.sqrt(np.nansum(w * (values - m) ** 2) / wsum))

def weighted_quantiles(values, w, qs):
    order = np.argsort(values)
    v, ww = values[order], w[order]
    cw = np.cumsum(ww)
    if cw[-1] <= 0:
        return [float("nan")] * len(qs)
    cw = cw / cw[-1]
    return [float(np.interp(q, cw, v)) for q in qs]

# CHANNEL_METRICS schema order: container_id, channel_id, value_type, sample_count,
# nan_ratio, begin_s, end_s, duration_ms, original_sample_count, original_sr,
# min, max, mean, std, pz1, pz10, pz90, pz99
metric_rows = []
for name in channel_names:
    cid = channel_meta[name]["channel_id"]
    arr = np.asarray(raw[name]["values"], dtype=np.float64)
    times, values = arr[:, 0], arr[:, 1]
    n = int(len(values))
    ss = SampleSeries.from_timestamps(times / 1e6, values)  # engine works in seconds
    dur = ss.durations()
    pz1, pz10, pz90, pz99 = weighted_quantiles(values, dur, [0.01, 0.10, 0.90, 0.99])
    metric_rows.append((
        CONTAINER_ID, cid,
        "DOUBLE",
        n,
        float(ss.nan_ratio()) if n else 0.0,
        float(times[0] / 1e6), float(times[-1] / 1e6),
        int((times[-1] - times[0]) / 1e3),
        n,
        float(np.mean(np.diff(times)) / 1e6) if n > 1 else 0.0,
        float(ss.min()), float(ss.max()), float(ss.mean()),
        weighted_std(values, dur),
        pz1, pz10, pz90, pz99,
    ))
channel_metrics_df = spark.createDataFrame(metric_rows, schema=S.CHANNEL_METRICS)
print(f"channel_metrics rows: {len(metric_rows)}")

In [ ]:
import datetime as dt
import pyspark.sql.types as T

def to_naive_utc(us):
    return dt.datetime.fromtimestamp(us / 1e6, tz=dt.timezone.utc).replace(tzinfo=None)

first = np.array([np.asarray(raw[n]["values"], dtype=np.float64)[:, 0].min() for n in channel_names])
last = np.array([np.asarray(raw[n]["values"], dtype=np.float64)[:, 0].max() for n in channel_names])
t_min, t_max = float(first.min()), float(last.max())

# Extend CONTAINER_METRICS with start_ts/stop_ts epoch-microsecond columns. These are the
# names the query engine's solver expects (SolverConfig.start_ts_col/stop_ts_col default to
# "start_ts"/"stop_ts"), so ContainerEvent and the default measurement_dimensions resolve
# them directly -- in the SAME microsecond unit as the channel timestamps (a plain rename of
# the start_dt TimestampType would cast to seconds and break alignment).
CONTAINER_METRICS_EXT = T.StructType(
    list(S.CONTAINER_METRICS.fields)
    + [T.StructField("start_ts", T.LongType()), T.StructField("stop_ts", T.LongType())]
)
container_metrics_df = spark.createDataFrame(
    [(CONTAINER_ID, to_naive_utc(t_min), to_naive_utc(t_max), int((t_max - t_min) / 1e3),
      len(channel_names), int(t_min), int(t_max))],
    schema=CONTAINER_METRICS_EXT,
)

ctags = [
    (CONTAINER_ID, "dataset", "A2D2 (Audi Autonomous Driving Dataset)"),
    (CONTAINER_ID, "city", CITY),
    (CONTAINER_ID, "recording", RECORDING),
    (CONTAINER_ID, "vehicle", "Audi"),
    (CONTAINER_ID, "source_url", BUS_URL),
    (CONTAINER_ID, "license", "CC BY-ND 4.0"),
    (CONTAINER_ID, "license_url", LICENSE_URL),
    (CONTAINER_ID, "attribution", "(c) Audi AG, A2D2, CC BY-ND 4.0"),
    (CONTAINER_ID, "camera_frontcenter_dir", cam_dir),
]
container_tags_df = spark.createDataFrame(ctags, schema=S.CONTAINER_TAGS)
print(f"recording {RECORDING}: {to_naive_utc(t_min)} .. {to_naive_utc(t_max)} UTC")

In [ ]:
writes = {
    "container_metrics": container_metrics_df,
    "container_tags": container_tags_df,
    "channel_metrics": channel_metrics_df,
    "channel_tags": channel_tags_df,
    "channels": channels_df,
}
# Idempotent per-container append: each drive is one container, so the three drives coexist in
# the same silver tables. Delete this container's rows (if the table exists) then append; the
# table is created on first write. (Re-running one drive only touches its own container_id.)
for name, df in writes.items():
    full = f"{pfx}_{name}"
    if spark.catalog.tableExists(full):
        spark.sql(f"DELETE FROM {full} WHERE container_id = {CONTAINER_ID}")
        df.write.mode("append").saveAsTable(full)
    else:
        df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(full)
    print(f"  wrote {full} (container {CONTAINER_ID})")
print("Bus silver-layer ingestion complete.")

## Camera images → parallel untar (PNG + timestamp)

The front-center camera tar is **98 GB**. A plain (uncompressed) tar is **512-byte-block
aligned** and self-describing, so we extract it **index-free, in parallel** with Spark:

1. Probe the tar's total size (one Range request) and split it into **`extract_partitions`
   byte-range chunks** (default **32**).
2. Each Spark task opens **one streaming HTTP Range read** of its chunk, aligns to the first
   valid tar header in the range, then walks headers **inline as the bytes stream past** —
   writing every PNG to the volume and parsing each JSON's `cam_tstamp`. No offset index, no
   per-header round-trips, no stragglers; it is bandwidth-bound and splits evenly.
3. Frames are assembled into `*_camera_frames`, **restricted to the bus-signal time window**
   `[t_min, t_max]` (frames outside it have no bus data to correlate), so they join to the bus
   signals by time. (All PNGs are written to the volume; only in-window frames are registered.)

Note: this works precisely because the archive is an **uncompressed `.tar`** — a `.tar.gz`
could not be split/streamed in parallel.

In [ ]:
import pyspark.sql.types as T
import pyspark.sql.functions as F

def _tar_total_size(url):
    """Total tar length via a 1-byte Range probe (Content-Range '.../<total>'); Content-Length fallback."""
    with requests.Session() as s:
        r = s.get(url, headers={"Range": "bytes=0-0"}, timeout=120)
        r.raise_for_status()
        cr = r.headers.get("Content-Range", "")
        if "/" in cr and cr.rsplit("/", 1)[-1].isdigit():
            return int(cr.rsplit("/", 1)[-1])
        cl = r.headers.get("Content-Length")
        if cl and cl.isdigit():
            return int(cl)
    raise RuntimeError("could not determine tar size")

EXTRACT_OUT_SCHEMA = T.StructType([
    T.StructField("frame_id", T.LongType()),
    T.StructField("png_path", T.StringType()),
    T.StructField("cam_tstamp_us", T.LongType()),
])

CAMERA_FRAMES_SCHEMA = T.StructType([
    T.StructField("container_id", T.LongType(), False),
    T.StructField("frame_id", T.IntegerType(), False),
    T.StructField("cam_tstamp_us", T.LongType()),
    T.StructField("cam_dt", T.TimestampType()),
    T.StructField("png_path", T.StringType()),
])

# Index-free parallel untar: a plain (uncompressed) tar is 512-byte-block aligned and self-
# describing, so we split it into byte-range chunks and let each Spark task stream its range
# once, parse tar headers inline, write every PNG to the volume, and read each JSON's
# cam_tstamp. No offset index, no per-header round-trips, no stragglers. Each member is owned
# by the chunk containing its header start (the boundary member is finished from the same open
# stream); duplicates at boundaries are harmless (deduped by the groupBy below).
def _make_extractor(url, cam, skip):
    def extract_chunk(iterator):
        import requests as _rq, pandas as _pd, json as __json, re as _re, os as _os
        _FR = _re.compile(r"_(\d{9})\.(png|jso)")
        ZERO = b"\x00" * 512
        def _oct(b):
            b = b.split(b"\x00")[0].strip()
            return int(b, 8) if b else 0
        def _csum(h):
            return len(h) >= 512 and _oct(h[148:156]) == sum(h[:148]) + sum(h[156:512]) + 32 * 8
        for pdf in iterator:
            out = []
            for _, c in pdf.iterrows():
                start, end = int(c["start"]), int(c["end"])
                resp = _rq.get(url, headers={"Range": f"bytes={start}-"}, stream=True, timeout=600)
                resp.raise_for_status()
                it = resp.iter_content(chunk_size=1024 * 1024)
                buf = bytearray()
                def rd(n):
                    while len(buf) < n:
                        try:
                            buf.extend(next(it))
                        except StopIteration:
                            break
                    take = bytes(buf[:n]); del buf[:n]; return take
                try:
                    off = start
                    if start != 0:
                        aligned, scanned = False, 0
                        while scanned <= 8 * 1024 * 1024:
                            hdr = rd(512)
                            if len(hdr) < 512:
                                break
                            if hdr[257:262] == b"ustar" and _csum(hdr):
                                aligned = True; break
                            off += 512; scanned += 512
                        if not aligned:
                            continue
                    else:
                        hdr = rd(512)
                    while True:
                        if len(hdr) < 512 or hdr == ZERO or not _csum(hdr):
                            break
                        if off >= end:
                            break
                        typ = hdr[156:157]; size = _oct(hdr[124:136])
                        padded = ((size + 511) // 512) * 512
                        name = None
                        if typ == b"L":
                            name = rd(padded)[:size].split(b"\x00")[0].decode("latin1").split("/")[-1]
                            off += 512 + padded
                            hdr = rd(512)
                            if len(hdr) < 512 or not _csum(hdr):
                                break
                            typ = hdr[156:157]; size = _oct(hdr[124:136])
                            padded = ((size + 511) // 512) * 512
                        payload = rd(padded)[:size]
                        if typ != b"5":
                            if name is None:
                                name = hdr[0:100].split(b"\x00")[0].decode("latin1").split("/")[-1]
                            m = _FR.search(name)
                            if m:
                                fid = int(m.group(1))
                                if ".png" in name:
                                    pp = f"{cam}/{name}"
                                    if not (skip and _os.path.exists(pp)):
                                        with open(pp, "wb") as fo:
                                            fo.write(payload)
                                    out.append((fid, pp, None))
                                elif ".jso" in name:
                                    try:
                                        ts = int(__json.loads(payload.decode("latin1"))["cam_tstamp"])
                                    except Exception:
                                        ts = None
                                    out.append((fid, None, ts))
                        off += 512 + padded
                        hdr = rd(512)
                finally:
                    resp.close()
            if out:
                yield _pd.DataFrame(out, columns=["frame_id", "png_path", "cam_tstamp_us"])
    return extract_chunk

if not DOWNLOAD_IMAGES:
    print("download_images=false -> skipping camera untar.")
elif (SKIP_IF_PRESENT and spark.catalog.tableExists(f"{pfx}_camera_frames")
      and spark.read.table(f"{pfx}_camera_frames").where(f"container_id = {CONTAINER_ID}").limit(1).count() > 0):
    print(f"container {CONTAINER_ID} already in {pfx}_camera_frames -> skipping camera untar.")
else:
    os.makedirs(cam_dir, exist_ok=True)
    total = _tar_total_size(CAM_URL)
    chunk = ((total + EXTRACT_PARTITIONS * 512 - 1) // (EXTRACT_PARTITIONS * 512)) * 512
    chunks = []
    for i in range(EXTRACT_PARTITIONS):
        s0 = i * chunk
        if s0 >= total:
            break
        chunks.append((i, int(s0), int(min(s0 + chunk, total))))
    print(f"Untarring {total/1e9:.1f} GB across {len(chunks)} parallel tasks -> {cam_dir} ...")
    chunks_df = spark.createDataFrame(chunks, "chunk_id int, start long, end long")
    raw_df = chunks_df.repartition(len(chunks)).mapInPandas(
        _make_extractor(CAM_URL, cam_dir, SKIP_IF_PRESENT), EXTRACT_OUT_SCHEMA)
    frames_df = (raw_df.groupBy("frame_id")
                 .agg(F.first("png_path", ignorenulls=True).alias("png_path"),
                      F.first("cam_tstamp_us", ignorenulls=True).alias("cam_tstamp_us"))
                 .where(f"cam_tstamp_us BETWEEN {int(t_min)} AND {int(t_max)}"))
    camera_frames_df = frames_df.select(
        F.lit(CONTAINER_ID).cast("long").alias("container_id"),
        F.col("frame_id").cast("int"),
        F.col("cam_tstamp_us").cast("long"),
        (F.col("cam_tstamp_us") / 1e6).cast("timestamp").alias("cam_dt"),
        F.col("png_path"))
    full_cf = f"{pfx}_camera_frames"
    if spark.catalog.tableExists(full_cf):
        spark.sql(f"DELETE FROM {full_cf} WHERE container_id = {CONTAINER_ID}")
        camera_frames_df.write.mode("append").saveAsTable(full_cf)
    else:
        camera_frames_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_cf)
    n = spark.read.table(full_cf).where(f"container_id = {CONTAINER_ID}").count()
    tot = spark.read.table(full_cf).count()
    print(f"Untar complete. Registered {n:,} in-window frames for container {CONTAINER_ID} "
          f"({tot:,} total across containers); window {to_naive_utc(t_min)} .. {to_naive_utc(t_max)} UTC; PNGs in {cam_dir}.")

## Validation — TSAL round-trip and image↔bus time-join

In [ ]:
from databricks.sdk import WorkspaceClient
from impulse_reporting.core.report import Report
from impulse_reporting.core.page import Page
from impulse_reporting.events.basic_event import BasicEvent
from impulse_reporting.aggregations.histogram import HistogramDuration
import pyspark.sql.functions as F

config = {
    "source": {
        "container_metrics_table": f"{pfx}_container_metrics",
        "container_tags_table": f"{pfx}_container_tags",
        "channel_metrics_table": f"{pfx}_channel_metrics",
        "channel_tags_table": f"{pfx}_channel_tags",
        "channels_uri": f"{pfx}_channels",
    },
    "unity_sink": {"catalog": CATALOG, "schema": SCHEMA, "table_prefix": TABLE_PREFIX},
    "query_engine": {"solver": "DeltaSolver", "data_type": "RAW"},
    "measurement_dimensions": ["container_id"],
}
report = Report(name="a2d2_validation", spark=spark, workspace_client=WorkspaceClient(), config=config)
db = report.get_db()

# Validation (same pattern as getting_started.ipynb): a 1D duration histogram of
# vehicle speed, scoped to a "vehicle moving" event, persisted to the gold star schema.
veh_speed = db.query.channel(channel_name="vehicle_speed")

moving = BasicEvent(name="vehicle_moving", expr=veh_speed > 0, desc="vehicle speed > 0 km/h")
report.add_event(moving)

page = Page(page_number=1)
page.add_aggregation(
    HistogramDuration(
        name="vehicle_speed_distribution",
        base_expr=veh_speed,
        bins=[float(x) for x in range(0, 161, 10)],
        event=moving,
        channel_name="vehicle_speed",
        bins_unit="km/h",
        values_unit="s",
    )
)
report.add_page(page)

report.determine_report()
report.persist_results()

print("Gold tables written. Duration-weighted vehicle-speed distribution:")
display(
    spark.read.table(f"{pfx}_histogram_fact")
    .groupBy("bin_name", "lower_bound")
    .agg(F.sum("hist_value").alias("duration_us"))
    .orderBy("lower_bound")
)

In [ ]:
# Match sampled camera frames to bus data by nearest timestamp (uses cam_tstamp).
if DOWNLOAD_IMAGES and spark.catalog.tableExists(f"{pfx}_camera_frames"):
    sp_cid = (spark.read.table(f"{pfx}_channel_tags")
              .where("key = 'channel_name' AND value = 'vehicle_speed'")
              .select("channel_id").first()[0])
    sp = (spark.read.table(f"{pfx}_channels").where(f"channel_id = {sp_cid}")
          .select("timestamp", "value").orderBy("timestamp").toPandas())
    cf = spark.read.table(f"{pfx}_camera_frames").orderBy("cam_tstamp_us").toPandas()

    idx = np.clip(np.searchsorted(sp["timestamp"].values, cf["cam_tstamp_us"].values), 0, len(sp) - 1)
    cf["vehicle_speed_kmh"] = sp["value"].values[idx]
    display(spark.createDataFrame(cf[["frame_id", "cam_dt", "vehicle_speed_kmh", "png_path"]]))

    # Render one sample frame (best-effort).
    try:
        import matplotlib.pyplot as plt
        import matplotlib.image as mpimg
        p = cf["png_path"].iloc[len(cf) // 2]
        plt.figure(figsize=(10, 6)); plt.imshow(mpimg.imread(p)); plt.axis("off")
        plt.title(f"frame {cf['frame_id'].iloc[len(cf) // 2]} @ {cf['cam_dt'].iloc[len(cf) // 2]} UTC")
        plt.show()
    except Exception as e:
        print("image preview skipped:", e)
else:
    print("No camera_frames table to join (images not downloaded).")

## Cleanup (optional)
Set the **drop_created_tables** widget to `true` and run the next cell to remove the tables
created by this notebook. Extracted images on the volume are left in place.

In [ ]:
if dbutils.widgets.get("drop_created_tables") == "true":
    # Drops everything this notebook created: silver tables, the gold star-schema tables
    # written by persist_results(), and camera_frames -- i.e. all {TABLE_PREFIX}_* tables.
    tbls = spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA} LIKE '{TABLE_PREFIX}_*'").collect()
    for r in tbls:
        spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.{r['tableName']}")
        print(f"  dropped {CATALOG}.{SCHEMA}.{r['tableName']}")
    print(f"Dropped {len(tbls)} {TABLE_PREFIX}_* tables. "
          f"(Volume + extracted images left in place; remove manually if desired.)")
else:
    print("drop_created_tables=false -> nothing dropped.")